# 05 - Event Integration (PRO ⚡)

Integración robusta de eventos con agregación por intervalo y validaciones.

In [1]:

import pandas as pd
from pathlib import Path

base_path = Path(r'C:/Users/rroman/OneDrive - Empresa Eléctrica de Guatemala, S.A. (EEGSA)/Documents/Unidad/2026/Capacitación/Data Science y Machine Learning/proyecto final/data')
input_path = base_path / 'processed' / 'dataset_features.parquet'
input_events_path = base_path / 'raw' / 'events_voltage_outage_0000.txt'
output_path = base_path / 'processed' / 'dataset_with_events.parquet'

# cargar features
df = pd.read_parquet(input_path)

print("Shape features:", df.shape)
df.head()


Shape features: (10258370, 34)


,Meter ID,Datetime,CurrentPhaseA,CurrentPhaseB,CurrentPhaseC,CurrentPhaseN,VoltagePhaseA,VoltagePhaseB,VoltagePhaseC,VoltageSag,...,Sag_duration_sec,Swell_duration_sec,Sag_pct,Swell_pct,Disturbance_score,hour,dayofweek,Voltage_drop,Current_spike,Voltage_event
0,F-74749,2026-02-09 00:15:00,0.0923,0.0924,0.0846,0.0,114.7291,113.1553,111.8357,0.0,...,0.0,0.0,0.0,0.0,0.0,0,0,NaN,NaN,0
1,F-74749,2026-02-09 00:30:00,0.0960,0.1010,0.0849,0.0,114.4624,113.5427,111.6652,0.0,...,0.0,0.0,0.0,0.0,0.0,0,0,-0.016600,0.004200,0
2,F-74749,2026-02-09 00:45:00,0.0842,0.0880,0.0746,0.0,114.3094,113.4678,111.4966,0.0,...,0.0,0.0,0.0,0.0,0.0,0,0,-0.132167,-0.011700,0
3,F-74749,2026-02-09 01:00:00,0.0983,0.1055,0.0884,0.0,114.3358,113.4620,111.5659,0.0,...,0.0,0.0,0.0,0.0,0.0,1,0,0.029967,0.015133,0
4,F-74749,2026-02-09 01:15:00,0.0883,0.0867,0.0818,0.0,114.3860,113.5381,111.5726,0.0,...,0.0,0.0,0.0,0.0,0.0,1,0,0.044333,-0.011800,0


In [2]:
df_test = pd.read_csv(
    input_events_path,
    nrows=5,
    header=None,
    engine="python"
)

print(df_test)

                                                   0
0                                        H~Event~1~1
1  D~Meter~P-96358~0001931090~1350383559~3~Endpoi...
2  D~Meter~F-86284~0000094873~1355596352~3~Endpoi...
3  D~Meter~F-91108~1724037~1355727867~3~Endpoint ...
4  D~Meter~Z-93217~0001867880~1358192520~3~Endpoi...


In [3]:
chunks = []

chunk_size = 100000

with open(input_events_path, "r", encoding="utf-8", errors="ignore") as f:
    buffer = []
    
    for i, line in enumerate(f):
        buffer.append(line.strip())
        
        if len(buffer) == chunk_size:
            chunk = pd.DataFrame(buffer, columns=["raw"])
            
            # 🔥 split manual seguro
            chunk = chunk["raw"].str.split("~", expand=True)
            
            # limpiar espacios
            chunk = chunk.apply(lambda x: x.str.strip() if x.dtype == "object" else x)
            
            # filtros
            chunk = chunk[chunk[0] == "D"]
            chunk = chunk[chunk[1] == "Meter"]
            
            chunks.append(chunk)
            buffer = []
    
    # último chunk
    if buffer:
        chunk = pd.DataFrame(buffer, columns=["raw"])
        chunk = chunk["raw"].str.split("~", expand=True)
        chunk = chunk.apply(lambda x: x.str.strip() if x.dtype == "object" else x)
        chunk = chunk[chunk[0] == "D"]
        chunk = chunk[chunk[1] == "Meter"]
        chunks.append(chunk)

df_events = pd.concat(chunks, ignore_index=True)

df_events.columns = [
    "Type","Device","Meter ID","Premise","Serial",
    "Code","Event Name","Event Time","Empty1","Description","Event Time 2"
]

print("Eventos cargados:", df_events.shape)


Eventos cargados: (1952397, 11)


In [4]:

# parse datetime
df_events["Datetime"] = pd.to_datetime(
    df_events["Event Time"],
    format="%m%d%Y%I%M%S%p",
    errors="coerce"
)

# bucket 15 min
df_events["Datetime_15m"] = df_events["Datetime"].dt.floor("15min")


## 🔥 Agregación por intervalo

In [5]:

df_events_agg = df_events.groupby(
    ["Meter ID", "Datetime_15m"]
).agg({
    "Event Name": lambda x: list(x)
}).reset_index()

df_events_agg["event_count"] = df_events_agg["Event Name"].apply(len)

df_events_agg["has_event"] = (df_events_agg["event_count"] > 0).astype(int)

df_events_agg["has_outage"] = df_events_agg["Event Name"].apply(
    lambda x: any("Outage" in str(e) or "Power Down" in str(e) for e in x)
).astype(int)

df_events_agg.head()


,Meter ID,Datetime_15m,Event Name,event_count,has_event,has_outage
0,,2026-02-20 07:30:00,[Endpoint Power Restore],1,1,0
1,,2026-03-05 13:00:00,[Endpoint Power Restore],1,1,0
2,,2026-03-07 11:45:00,"[Endpoint Power Outage, Endpoint Power Restore]",2,1,1
3,,2026-03-07 12:00:00,"[Endpoint Power Outage, Endpoint Power Restore]",2,1,1
4,,2026-03-10 07:45:00,[Endpoint Power Restore],1,1,0


## 🔗 Merge con dataset

In [6]:

df_final = df.merge(
    df_events_agg,
    left_on=["Meter ID","Datetime"],
    right_on=["Meter ID","Datetime_15m"],
    how="left"
)

# limpiar nulos
df_final["has_event"] = df_final["has_event"].fillna(0)
df_final["has_outage"] = df_final["has_outage"].fillna(0)
df_final["event_count"] = df_final["event_count"].fillna(0)

df_final.head()


,Meter ID,Datetime,CurrentPhaseA,CurrentPhaseB,CurrentPhaseC,CurrentPhaseN,VoltagePhaseA,VoltagePhaseB,VoltagePhaseC,VoltageSag,...,hour,dayofweek,Voltage_drop,Current_spike,Voltage_event,Datetime_15m,Event Name,event_count,has_event,has_outage
0,F-74749,2026-02-09 00:15:00,0.0923,0.0924,0.0846,0.0,114.7291,113.1553,111.8357,0.0,...,0,0,NaN,NaN,0,NaT,NaN,0.0,0.0,0.0
1,F-74749,2026-02-09 00:30:00,0.0960,0.1010,0.0849,0.0,114.4624,113.5427,111.6652,0.0,...,0,0,-0.016600,0.004200,0,NaT,NaN,0.0,0.0,0.0
2,F-74749,2026-02-09 00:45:00,0.0842,0.0880,0.0746,0.0,114.3094,113.4678,111.4966,0.0,...,0,0,-0.132167,-0.011700,0,NaT,NaN,0.0,0.0,0.0
3,F-74749,2026-02-09 01:00:00,0.0983,0.1055,0.0884,0.0,114.3358,113.4620,111.5659,0.0,...,1,0,0.029967,0.015133,0,NaT,NaN,0.0,0.0,0.0
4,F-74749,2026-02-09 01:15:00,0.0883,0.0867,0.0818,0.0,114.3860,113.5381,111.5726,0.0,...,1,0,0.044333,-0.011800,0,NaT,NaN,0.0,0.0,0.0


## ✅ Validaciones

In [7]:

print("Shape final:", df_final.shape)

print("\nEventos detectados:")
print(df_final["has_event"].value_counts())

print("\nOutages detectados:")
print(df_final["has_outage"].value_counts())


Shape final: (10258370, 39)

Eventos detectados:
has_event
0.0    9914807
1.0     343563
Name: count, dtype: int64

Outages detectados:
has_outage
0.0    10254074
1.0        4296
Name: count, dtype: int64


In [8]:

print("\nDistribución por hora:")
print(df_final.groupby(df_final["Datetime"].dt.hour)["has_outage"].mean())



Distribución por hora:
Datetime
0     0.000053
1     0.000103
2     0.000016
3     0.000058
4     0.000053
5     0.000151
6     0.000219
7     0.000278
8     0.000527
9     0.000492
10    0.000949
11    0.001511
12    0.000687
13    0.000588
14    0.000769
15    0.001991
16    0.000708
17    0.000076
18    0.000340
19    0.000208
20    0.000073
21    0.000106
22    0.000087
23    0.000049
Name: has_outage, dtype: float64


In [9]:

print("\nRelación voltaje vs outage:")
print(df_final.groupby("has_outage")["Voltage_avg"].mean())



Relación voltaje vs outage:
has_outage
0.0    128.417455
1.0    125.082590
Name: Voltage_avg, dtype: float64


In [11]:
df_final.groupby("has_outage")[[
    "Voltage_unbalance",
    "Sag_pct",
    "Current_avg"
]].mean()

,Voltage_unbalance,Sag_pct,Current_avg
has_outage,,,
0.0,34.377765,0.087796,0.543271
1.0,39.946198,0.104805,0.694562


## 💾 Guardar dataset

In [10]:

df_final.to_parquet(output_path, index=False)

print("✅ Dataset con eventos listo")


✅ Dataset con eventos listo
